# CUAD — Final Preprocessing Notebook (Version 4)

**Goal:** Prepare CUAD legal contracts for **multi-label clause classification**. We want a BERT model to predict which of the **41 clause categories** appear in each piece of contract text.

## What we decided

| Item | Our choice | Why |
|---|---|---|
| Pretrained tokenizer | Hugging Face `AutoTokenizer.from_pretrained("bert-base-uncased")` | Loads the tokenizer that matches the selected BERT model |
| Tokenization algorithm | **WordPiece** | Breaks text into subword tokens; we did not create our own vocabulary |
| Fast tokenizer | `use_fast=True` | Gives character offsets so we can match tokens to CUAD answers |
| Maximum window | **512 BERT tokens**, including `[CLS]` and `[SEP]` | Fits BERT's input limit |
| Window overlap | **256 content tokens** | Keeps extra context around window boundaries |
| Contract split | **326 train / 82 validation / 102 test** | Prevents data leakage between contracts |
| Prediction task | **41-label multi-label classification** | A window can contain several clause categories |
| Labeling policy | **Per-Category Label Masking (PCLM)** | Keeps the text while ignoring uncertain category labels |

### Quick definitions

- **Annotation / answer span:** A passage marked in CUAD as evidence for a particular clause category.
- **Chunk / window:** A smaller section of a long contract that we give to BERT.
- **Token:** A word, part of a word, or punctuation produced by the tokenizer—not always one whole word.
- **WordPiece:** BERT's method for splitting unfamiliar words into smaller pieces, such as `appoint` + `##s`.
- **Sliding window:** Move a fixed-size token window through the contract until all text is processed.
- **Overlap / stride:** Tokens repeated between consecutive windows; our overlap is 256 tokens.
- **Offset mapping:** The original character positions represented by each token.
- **Multi-hot vector:** A list of 41 zeros and ones; more than one category may be `1`.
- **Attention mask:** `1` marks a real input token and `0` marks padding added later for batching.
- **Label mask:** `1` means we can train on that category; `0` means ignore its loss for this window.
- **Annotation coverage:** The percentage of annotated answer spans fully contained in at least one usable window.

### How we reached Version 4

Versions 1–2 used **word-based overlapping chunks**, but boundaries could cut annotated answers. Version 3 moved boundaries to protect answers; its earlier dataset had **13,003 training-source chunks**, of which **1,103 exceeded 512 BERT tokens**. Our first Version 4 kept the BERT limit but removed whole windows with partial answers, reducing coverage. Our **final Version 4** keeps all token-sized windows and masks only uncertain categories.

This notebook contains **one final preprocessing pipeline**. The earlier versions are experiments, not extra steps we need to run. We are using a **pretrained tokenizer**, but we have **not trained the BERT classifier yet**.

## 1. Load the tokenizer and choose our settings

**Why Hugging Face and WordPiece both appear:** We chose the checkpoint name `bert-base-uncased`; Hugging Face `AutoTokenizer` automatically loads the corresponding **pretrained WordPiece tokenizer**. *Auto* selects the tokenizer class for our chosen checkpoint—it does not choose a different model. *Uncased* means it generally lowercases the text.

- **`MAX_LENGTH = 512`:** Maximum tokens per model input, including two special tokens (`[CLS]` and `[SEP]`).
- **`STRIDE = 256`:** Number of content tokens repeated between adjacent windows.
- **`use_fast=True`:** Needed to return original-character positions for label matching.
- **`truncation=True` together with `return_overflowing_tokens=True`:** Makes multiple windows instead of throwing away the remainder of a long contract.

The warning that PyTorch is unavailable only affects **model training**, not this tokenization-only notebook.

In [1]:
# Uncomment only if needed:
# %pip install transformers scikit-learn

import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

BASE_DIR = Path("../data/data")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = PROCESSED_DIR / "tokenized_bert_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
STRIDE = 256               # Number of overlapping BERT content tokens
RANDOM_STATE = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "Offset mapping requires a fast tokenizer"
assert MAX_LENGTH <= tokenizer.model_max_length
assert 0 < STRIDE < (MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False))

print("Tokenizer:", type(tokenizer).__name__)
print("Algorithm:", type(tokenizer.backend_tokenizer.model).__name__)
print("Model maximum:", tokenizer.model_max_length)
print("Window maximum:", MAX_LENGTH, "token overlap:", STRIDE)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Tokenizer: BertTokenizer
Algorithm: WordPiece
Model maximum: 512
Window maximum: 512 token overlap: 256


## 2. Split contracts before making windows

CUAD provides **408 training-source contracts** and **102 separate test contracts**. We keep the original test set untouched and use `train_test_split(test_size=0.2, random_state=42)` to divide the 408 into **326 training** and **82 validation** contracts.

That is **63.92% train, 16.08% validation, and 20% test** across all 510 contracts—approximately **64/16/20**. The small difference is because contracts cannot be split into fractional pieces.

**Contract-level split rule:** All windows from one contract must stay in the same split. We split **first**, then tokenize, to avoid data leakage. `random_state=42` makes the split reproducible. When comparing models, the actual contract titles must match, not just the totals.

In [2]:
def read_contracts(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}. Check the notebook working directory.")
    return json.loads(path.read_text(encoding="utf-8"))["data"]

source_train_contracts = read_contracts(BASE_DIR / "train_separate_questions.json")
test_contracts = read_contracts(BASE_DIR / "test.json")
categories = json.loads((PROCESSED_DIR / "cuad_categories.json").read_text(encoding="utf-8"))

assert len(source_train_contracts) == 408 and len(test_contracts) == 102, "Unexpected official data split"
assert len(categories) == 41 and len(set(categories)) == 41, "Expected 41 unique CUAD categories"

train_contracts, validation_contracts = train_test_split(
    source_train_contracts, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)
splits = {
    "train": train_contracts,
    "validation": validation_contracts,
    "test": test_contracts,
}

title_sets = {name: {c["title"] for c in items} for name, items in splits.items()}
assert all(len(title_sets[name]) == len(items) for name, items in splits.items()), "Duplicate titles in split"
assert not (title_sets["train"] & title_sets["validation"])
assert not (title_sets["train"] & title_sets["test"])
assert not (title_sets["validation"] & title_sets["test"])
assert (len(train_contracts), len(validation_contracts), len(test_contracts)) == (326,82,102)

label2id = {name: i for i, name in enumerate(categories)}
id2label = {str(i): name for name, i in label2id.items()}
print("Contract splits:", {name: len(items) for name, items in splits.items()})
print("Labels:", len(label2id))

Contract splits: {'train': 326, 'validation': 82, 'test': 102}
Labels: 41


## 3. Align CUAD answers with the original text

CUAD provides each answer's `answer_start` position **inside its paragraph**. We join a contract's paragraphs with **one space**, convert the paragraph offsets into whole-contract character positions, and confirm that `text[start:end]` exactly matches the annotated answer.

**Why this matters:** If an answer position is wrong, we could assign a clause label to the wrong window. We stop the notebook if an annotation or category does not match.

**Category rule:** We extract the category from the CUAD question ID and remove an ending number such as `_0`; categories are checked against the fixed list of 41.

In [3]:
def contract_text(contract):
    return " ".join(p["context"] for p in contract["paragraphs"])

def get_answer_spans(contract):
    spans = []
    offset = 0
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            if qa.get("is_impossible", False):
                continue
            for answer in qa["answers"]:
                start = offset + answer["answer_start"]
                end = start + len(answer["text"])
                spans.append((start, end, qa["id"], answer["text"]))
        offset += len(context) + 1
    return spans

def category_from_id(qid):
    category = qid.rsplit("__", 1)[-1]
    suffix = category.rsplit("_", 1)
    return suffix[0] if len(suffix) == 2 and suffix[1].isdigit() else category

invalid = []
unknown_categories = []
for split_name, items in splits.items():
    for c in items:
        text = contract_text(c)
        for start, end, qid, expected_answer in get_answer_spans(c):
            if text[start:end] != expected_answer:
                invalid.append((split_name, c["title"], qid))
            if category_from_id(qid) not in label2id:
                unknown_categories.append((split_name, qid, category_from_id(qid)))
print("Invalid annotation offsets:", len(invalid))
print("Unknown label categories:", len(unknown_categories))
assert not invalid, f"Fix offset mismatch first: {invalid[:3]}"
assert not unknown_categories, f"Check label mapping first: {unknown_categories[:3]}"

Invalid annotation offsets: 0
Unknown label categories: 0


## 4. Create token windows and apply our labeling rules

We use the Hugging Face tokenizer on the **complete contract**, producing consecutive windows of **at most 512 tokens** with **256 overlapping tokens**. `return_overflowing_tokens=True` returns every window, while `return_offsets_mapping=True` identifies which original characters appear in each one.

### What is masking?

**Masking does not remove contract text.** It tells the training loss to **skip one uncertain category** in a window, while still learning from all the other categories.

We call our policy **Per-Category Label Masking (PCLM)**. This is a descriptive name for the rules in our project, not a claim that we created a new ML algorithm.

### Our five labeling rules

| Rule | One-line definition | `label` | `label_mask` |
|---|---|---:|---:|
| **1. Complete Answer Rule** | The full annotated answer appears inside the window: learn a positive label. | 1 | 1 |
| **2. Partial Answer Rule** | Only part of the answer appears, with no complete answer of that category: ignore this category. | 0 | 0 |
| **3. No Annotation Rule** | No answer of this category overlaps the window: use it as a negative training label. | 0 | 1 |
| **4. Complete Answer Priority Rule** | A full answer and partial answer of the **same** category appear: the complete positive answer wins. | 1 | 1 |
| **5. Window Retention Rule** | Keep **every** window even if some of its categories are masked. | Per category | Per category |

**Example:** One window contains a complete **Parties** answer and half of a **License Grant** answer. Parties is `(label=1, mask=1)`; License Grant is `(label=0, mask=0)`; a category with no overlapping annotation is `(label=0, mask=1)`. The text is unchanged.

### What the two vectors mean

- **`labels`:** 41 zeros and ones indicating complete annotated answers by category.
- **`label_mask`:** 41 zeros and ones indicating which category positions are safe to use in the loss.

A mask of `0` means **unknown for this window**, not **absent from the contract**. A mask of `1` with label `0` is a negative according to CUAD's annotations, which may not capture every possible real-world clause.

**Important design decision:** Our window boundaries do **not** depend on known answers. We use annotations only to create supervision labels, so the same sliding windows can be applied to new contracts at prediction time.

In [4]:
def windows_for_contract(contract, split_name):
    text = contract_text(contract)
    spans = get_answer_spans(contract)
    if not text.strip():
        return [], 0, []

    encoded = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        add_special_tokens=True,
        padding=False,
    )

    rows = []
    inference_rows = []
    windows_with_partials = 0

    for win_i, (ids, attention_mask, offsets) in enumerate(zip(
        encoded["input_ids"], encoded["attention_mask"], encoded["offset_mapping"]
    )):
        assert len(ids) <= MAX_LENGTH, "BERT length limit violated"
        content_offsets = [(s, e) for s, e in offsets if e > s]
        if not content_offsets:
            continue

        begin = min(s for s, e in content_offsets)
        finish = max(e for s, e in content_offsets)

        # These are the SAME fixed windows we will use on unseen contracts.
        inference_row = {
            "contract_title": contract["title"],
            "window_id": win_i,
            "char_start": begin,
            "char_end": finish,
            "input_ids": ids,
            "attention_mask": attention_mask,
        }
        if "token_type_ids" in encoded:
            inference_row["token_type_ids"] = encoded["token_type_ids"][win_i]
        inference_rows.append(inference_row)

        full_indices = []
        full_categories = set()
        partial_categories = set()

        for span_i, (a, b, qid, answer_text) in enumerate(spans):
            if a < finish and begin < b:  # Answer overlaps this window.
                category = category_from_id(qid)
                if begin <= a and b <= finish:
                    full_indices.append(span_i)
                    full_categories.add(category)
                else:
                    partial_categories.add(category)

        if partial_categories:
            windows_with_partials += 1

        labels = [int(category in full_categories) for category in categories]

        # A full answer makes a positive label usable even if another
        # answer of the same category partly overlaps the window.
        label_mask = [
            int(category in full_categories or category not in partial_categories)
            for category in categories
        ]

        row = {
            "split": split_name,
            "contract_title": contract["title"],
            "window_id": win_i,
            "char_start": begin,
            "char_end": finish,
            "text": text[begin:finish],
            "input_ids": ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "label_mask": label_mask,
            "label_names": sorted(full_categories),
            "partial_label_names": sorted(partial_categories - full_categories),
            # For coverage auditing; the model does not use these fields.
            "covered_span_indices": full_indices,
        }
        if "token_type_ids" in encoded:
            row["token_type_ids"] = encoded["token_type_ids"][win_i]
        rows.append(row)

    assert len(rows) == len(inference_rows), "Labeled and inference windows should match"
    return rows, windows_with_partials, inference_rows


all_rows = {}
all_inference_rows = {}
partial_window_counts = {}

for name, items in splits.items():
    all_rows[name] = []
    all_inference_rows[name] = []
    partial_window_counts[name] = 0
    for contract in items:
        rows, partial_count, inference_rows = windows_for_contract(contract, name)
        all_rows[name].extend(rows)
        all_inference_rows[name].extend(inference_rows)
        partial_window_counts[name] += partial_count

    print(
        name,
        "saved windows:", len(all_rows[name]),
        "windows with at least one partial answer:", partial_window_counts[name],
        "ignored label positions:", sum(
            row["label_mask"].count(0) for row in all_rows[name]
        ),
    )

train saved windows: 14761 windows with at least one partial answer: 2852 ignored label positions: 2428
validation saved windows: 2937 windows with at least one partial answer: 542 ignored label positions: 475
test saved windows: 3816 windows with at least one partial answer: 741 ignored label positions: 597


## 5. Check token limits, label masks, and answer coverage

We run these checks before treating the dataset as ready:

- **Token limit:** Every `input_ids` sequence contains **512 or fewer tokens**.
- **Label structure:** Every window has **41 labels and 41 mask values**; positive labels are never masked.
- **Attention mask:** It has the same length as the token IDs. Padding can be added later when building batches.
- **Split isolation:** One contract belongs to only one dataset split.
- **Annotation coverage:** We count how many original answers appear **completely** in at least one window with a usable positive label.
- **Exceptions:** We save missing annotations instead of silently claiming 100% coverage.

Even with overlap, some answers may fall across window boundaries or be longer than the 510 available content tokens. **High annotation coverage measures preprocessing, not model classification accuracy.** The model still needs its own evaluation.

In [5]:
coverage_report = {}
missing_answers = []

for name, items in splits.items():
    rows_by_title = {}
    for row in all_rows[name]:
        rows_by_title.setdefault(row["contract_title"], []).append(row)

    total = covered = too_long = 0
    for contract in items:
        title = contract["title"]
        for span_i, (start, end, qid, answer) in enumerate(get_answer_spans(contract)):
            total += 1
            category = category_from_id(qid)
            category_i = label2id[category]

            found = any(
                span_i in row["covered_span_indices"]
                and row["labels"][category_i] == 1
                and row["label_mask"][category_i] == 1
                for row in rows_by_title.get(title, [])
            )
            if found:
                covered += 1
                continue

            # Count subwords without trying to pass an overlength input into BERT.
            token_count = len(tokenizer.tokenize(answer)) + tokenizer.num_special_tokens_to_add(pair=False)
            if token_count > MAX_LENGTH:
                too_long += 1
            missing_answers.append({
                "split": name,
                "contract_title": title,
                "qa_id": qid,
                "answer_start": start,
                "answer_end": end,
                "answer_tokens_with_specials": token_count,
                "reason": "answer_over_512" if token_count > MAX_LENGTH else "not_fully_contained_in_any_window",
            })

    rows = all_rows[name]
    coverage_report[name] = {
        "contracts": len(items),
        "annotated_answers": total,
        "answers_covered_by_windows": covered,
        "coverage_percent": round(100 * covered / total, 2) if total else None,
        "missing_answers": total - covered,
        "missing_because_answer_too_long": too_long,
        "windows_with_partial_annotations": partial_window_counts[name],
        "windows_excluded_due_to_partial": 0,
        "saved_windows": len(rows),
        "masked_label_positions": sum(row["label_mask"].count(0) for row in rows),
        "all_inference_windows": len(all_inference_rows[name]),
        "positive_windows": sum(any(row["labels"]) for row in rows),
        "no_positive_labels_windows": sum(not any(row["labels"]) for row in rows),
        "max_input_tokens": max((len(row["input_ids"]) for row in rows), default=0),
    }
    print(name, coverage_report[name])

assert all(len(row["input_ids"]) <= MAX_LENGTH
           for rows in all_rows.values() for row in rows)
assert all(len(row["labels"]) == len(categories)
           and len(row["label_mask"]) == len(categories)
           for rows in all_rows.values() for row in rows)
assert all(len(row["attention_mask"]) == len(row["input_ids"])
           for rows in all_rows.values() for row in rows)
assert all(
    all(label == 0 or mask == 1 for label, mask in zip(row["labels"], row["label_mask"]))
    for rows in all_rows.values() for row in rows
), "A positive label must never be ignored"
assert all(len(all_rows[name]) == len(all_inference_rows[name])
           for name in splits), "Every fixed window should be retained"
assert all_rows["train"] and all_rows["validation"] and all_rows["test"]
print("\nChecks passed: token lengths, 41 labels, masks, window counts, nonempty splits")
print("Uncovered annotation examples:", missing_answers[:3])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors


train {'contracts': 326, 'annotated_answers': 9175, 'answers_covered_by_windows': 9120, 'coverage_percent': 99.4, 'missing_answers': 55, 'missing_because_answer_too_long': 8, 'windows_with_partial_annotations': 2852, 'windows_excluded_due_to_partial': 0, 'saved_windows': 14761, 'masked_label_positions': 2428, 'all_inference_windows': 14761, 'positive_windows': 5780, 'no_positive_labels_windows': 8981, 'max_input_tokens': 512}
validation {'contracts': 82, 'annotated_answers': 2005, 'answers_covered_by_windows': 1996, 'coverage_percent': 99.55, 'missing_answers': 9, 'missing_because_answer_too_long': 4, 'windows_with_partial_annotations': 542, 'windows_excluded_due_to_partial': 0, 'saved_windows': 2937, 'masked_label_positions': 475, 'all_inference_windows': 2937, 'positive_windows': 1179, 'no_positive_labels_windows': 1758, 'max_input_tokens': 512}
test {'contracts': 102, 'annotated_answers': 2643, 'answers_covered_by_windows': 2631, 'coverage_percent': 99.55, 'missing_answers': 12, 'mi

## 6. Save the data and prepare for model training

We save the outputs as **JSONL** (one window per line) in `../data/processed/tokenized_bert_v4/`:

| Files | Why we save them |
|---|---|
| `train.jsonl`, `validation.jsonl`, `test.jsonl` | Windows with token IDs, attention masks, 41 targets, and 41 masks |
| `validation_all_windows.jsonl`, `test_all_windows.jsonl` | Unfiltered windows for realistic predictions on complete contracts |
| `validation_contract_gold.jsonl`, `test_contract_gold.jsonl` | True labels at the **whole-contract** level |
| `label2id.json`, `id2label.json` | The fixed category order for training and interpreting predictions |
| `split_contract_titles.json` | The exact reproducible contract-level split |
| `audit.json`, `missing_answers.jsonl` | Coverage numbers and cases the windows did not fully preserve |

We do **not** pad every example to 512 here. At training time, we can pad each batch to its longest item.

### Rule for the future BERT training notebook: Masked Binary Cross-Entropy

- **Binary cross-entropy:** Computes a classification error independently for each of our 41 categories.
- **Masked loss:** Multiplies each error by its `label_mask` value, so uncertain categories (`mask=0`) do not affect training.

```python
per_label_loss = torch.nn.functional.binary_cross_entropy_with_logits(
    logits, labels.float(), reduction="none"
)
loss = (per_label_loss * label_mask.float()).sum() / label_mask.float().sum().clamp_min(1)
```

The model receives `input_ids`, `attention_mask` and optional `token_type_ids`. **`label_mask` is used when computing the loss, not passed as a BERT input.** Without this masked loss, the uncertain categories would incorrectly be trained as negatives.

For evaluation, predict on **all fixed validation/test windows**, combine window predictions into contract-level predictions, and compare against the contract's true categories. Do not select evaluation windows using their known answers.

In [6]:
def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for item in rows:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

for name, rows in all_rows.items():
    write_jsonl(OUTPUT_DIR / f"{name}.jsonl", rows)

# Keep all fixed validation/test windows for contract-level predictions.
for name in ("validation", "test"):
    write_jsonl(OUTPUT_DIR / f"{name}_all_windows.jsonl", all_inference_rows[name])
    contract_gold = []
    for contract in splits[name]:
        present = {category_from_id(qid) for _, _, qid, _ in get_answer_spans(contract)}
        contract_gold.append({
            "contract_title": contract["title"],
            "labels": [int(category in present) for category in categories],
            "label_names": sorted(present),
        })
    write_jsonl(OUTPUT_DIR / f"{name}_contract_gold.jsonl", contract_gold)

write_jsonl(OUTPUT_DIR / "missing_answers.jsonl", missing_answers)
(OUTPUT_DIR / "label2id.json").write_text(json.dumps(label2id, indent=2), encoding="utf-8")
(OUTPUT_DIR / "id2label.json").write_text(json.dumps(id2label, indent=2), encoding="utf-8")
(OUTPUT_DIR / "split_contract_titles.json").write_text(
    json.dumps({name: sorted(titles) for name, titles in title_sets.items()}, indent=2),
    encoding="utf-8"
)

audit = {
    "model_name": MODEL_NAME,
    "tokenizer_type": type(tokenizer).__name__,
    "max_length": MAX_LENGTH,
    "stride_overlap_tokens": STRIDE,
    "label_policy": "complete_answer_positive__partial_category_masked__all_windows_retained",
    "contract_level_split_seed": RANDOM_STATE,
    "notes": [
        "Window boundaries are independent of known annotation positions.",
        "For a partial annotation, only that uncertain category is masked (unless another full answer gives a positive label for the same category).",
        "Unmasked binary cross-entropy is NOT appropriate; use label_mask in the model's loss function.",
        "Some annotations may remain uncovered because they span fixed-window boundaries or exceed the BERT limit.",
        "Use all fixed validation/test windows for contract-level evaluation, without gold filtering.",
        "Keep the original 102-contract test split untouched until the final evaluation.",
    ],
    "splits": coverage_report,
}
(OUTPUT_DIR / "audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("Saved output folder:", OUTPUT_DIR.resolve())
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

Saved output folder: /Users/rashipatel/Documents/Accenture Project /Accenture-1G-contract-review-challenge/data/processed/tokenized_bert_v4
Files: ['audit.json', 'id2label.json', 'label2id.json', 'missing_answers.jsonl', 'split_contract_titles.json', 'test.jsonl', 'test_all_windows.jsonl', 'test_contract_gold.jsonl', 'train.jsonl', 'validation.jsonl', 'validation_all_windows.jsonl', 'validation_contract_gold.jsonl']


In [7]:
# Results from this run, ready for our project notes and future slides.
print("\nCUAD FINAL VERSION 4 — PREPROCESSING RESULTS")
print("Tokenizer:", MODEL_NAME, "| Max input:", MAX_LENGTH, "| Overlap:", STRIDE)
for split_name in ("train", "validation", "test"):
    s = coverage_report[split_name]
    print(
        f"{split_name.title()}: {s['contracts']} contracts; "
        f"{s['saved_windows']} saved windows; "
        f"{s['windows_with_partial_annotations']} windows contain partial answers; "
        f"{s['masked_label_positions']} uncertain category positions masked; "
        f"{s['answers_covered_by_windows']}/{s['annotated_answers']} "
        f"answers covered ({s['coverage_percent']}%); "
        f"{s['missing_because_answer_too_long']} uncovered answers individually over 512 tokens"
    )
print("Results and files saved in:", OUTPUT_DIR.resolve())


CUAD FINAL VERSION 4 — PREPROCESSING RESULTS
Tokenizer: bert-base-uncased | Max input: 512 | Overlap: 256
Train: 326 contracts; 14761 saved windows; 2852 windows contain partial answers; 2428 uncertain category positions masked; 9120/9175 answers covered (99.4%); 8 uncovered answers individually over 512 tokens
Validation: 82 contracts; 2937 saved windows; 542 windows contain partial answers; 475 uncertain category positions masked; 1996/2005 answers covered (99.55%); 4 uncovered answers individually over 512 tokens
Test: 102 contracts; 3816 saved windows; 741 windows contain partial answers; 597 uncertain category positions masked; 2631/2643 answers covered (99.55%); 2 uncovered answers individually over 512 tokens
Results and files saved in: /Users/rashipatel/Documents/Accenture Project /Accenture-1G-contract-review-challenge/data/processed/tokenized_bert_v4


## 7. Results, limitations, and conclusion

### Results from our completed Version 4 run (November 11, 2026)

| Dataset | Contracts | Windows retained | Answers fully covered | Coverage |
|---|---:|---:|---:|---:|
| Training | 326 | 14,761 | 9,120 / 9,175 | **99.40%** |
| Validation | 82 | 2,937 | 1,996 / 2,005 | **99.55%** |
| Test | 102 | 3,816 | 2,631 / 2,643 | **99.55%** |

This run retained **all generated windows**. The number of uncertain category positions masked was **2,428** in training, **475** in validation, and **597** in test. **55** training annotations, **9** validation annotations, and **12** test annotations were not fully covered by a usable window; these are recorded in `missing_answers.jsonl`.

**These are recorded preprocessing results from a completed run, not model accuracy results.** When running the notebook again, use the newly printed audit as the source of truth.

### Our conclusion

We moved from word-based chunks (Versions 1–2), to annotation-aware boundary adjustments (Version 3), to **BERT-token-based sliding windows (final Version 4)**. The final method enforces the **512-token input limit**, uses **256-token overlap**, preserves the original contract split, and applies **Per-Category Label Masking** so partial answers do not make us discard otherwise useful windows.

**Limitations:** This method does not guarantee complete coverage of every annotation; some are too long or still cross window boundaries. The 256-token overlap is a tested setting for this run, not proof that it is optimal. Absence of a CUAD annotation is also not necessarily proof that a clause is absent in real text.

### Next step in our project

Use the saved dataset to **fine-tune a BERT multi-label sequence classifier** with masked binary cross-entropy. Select thresholds on validation contracts and evaluate on the untouched test contracts **after** model selection. Report micro-F1, macro-F1, precision, and recall.

**Short explanation for our presentation:** *We tried different chunking strategies and found that word-based chunks could split answers, while annotation-aware chunks could exceed BERT's limit. Our final approach uses Hugging Face WordPiece tokenization, 512-token overlapping windows, and per-category label masking. We measured annotation coverage before starting model training.*

**Reference:** [Hugging Face tokenizer documentation](https://huggingface.co/docs/transformers/en/main_classes/tokenizer).